In [ ]:
import sys

sys.path.append('../')
sys.path.append('../../src/')

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.metrics import recall_score, make_scorer, roc_auc_score
from preprocess import preprocess

specificity = make_scorer(recall_score, pos_label=0)

In [ ]:
dat = pd.read_parquet('../depression_all_features.parquet')
dat.shape

In [ ]:
dat = dat.pivot(index='birthid', columns='feature_name', values='value').reset_index()
dat.columns.name = ''
dat

In [ ]:
# remove the earliest and latest date to be consistent with the paper
cols_todo = [x for x in dat.columns if x.endswith('_has_rx') or x.endswith('_has_dx')]
cols_to_drop = []
for col in cols_todo:
    code = col[:col.index('_has')]
    cols_to_drop.append(f"{code}_earliest_day")
    cols_to_drop.append(f"{code}_latest_day")
dat = dat.drop(columns=cols_to_drop)
print(dat.shape)

In [ ]:
preprocess(dat)

In [ ]:
dat.head()

In [ ]:
# make sure there's no NaN except for the phq9_total_max and edinburgh_max
dat.isna().mean().sort_values(ascending=False)

In [ ]:
dat_tmp = dat[['F53_label', 'edinburgh_max', 'phq9_total_max']].copy()
dat_tmp

In [ ]:
dat_tmp = dat_tmp[(dat_tmp['edinburgh_max'].isna()==False) | (dat_tmp['phq9_total_max'].isna()==False)].copy()
dat_tmp

In [ ]:
dat_tmp['screen'] = np.where( (dat_tmp['edinburgh_max']>=10) | (dat_tmp['phq9_total_max']>=10), 1, 0)

In [ ]:
dat_tmp.groupby(['screen','F53_label'])['F53_label'].count()

### 1. do some simple run to see the baseline

In [ ]:
def print_best_score(clf):
    idx = clf.cv_results_['rank_test_roc_auc'].tolist().index(1)
    print(clf.best_params_, clf.best_score_)
    print(f"auc = {clf.cv_results_['mean_test_roc_auc'][idx]} +- {clf.cv_results_['std_test_roc_auc'][idx]}")
    print(f"sensitivity = {clf.cv_results_['mean_test_sensitivity'][idx]} +- {clf.cv_results_['std_test_sensitivity'][idx]}")
    print(f"specificity = {clf.cv_results_['mean_test_specificity'][idx]} +- {clf.cv_results_['std_test_specificity'][idx]}")

In [ ]:
clf = GridSearchCV(RandomForestClassifier(class_weight='balanced', random_state=42, n_estimators=200),
                   param_grid={'min_samples_split': [30,50,70,100,120,150,170,200,250,300,500]},
                   verbose=3,
                   scoring={'f1':'f1',
                            'roc_auc':'roc_auc',
                            'sensitivity':'recall',
                            'precision':'precision',
                            'specificity': specificity},
                   cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
                   refit='roc_auc',
                   n_jobs=4)

X = dat.drop(columns=['F53_label', 'edinburgh_max','phq9_total_max', 'birthid', 'target_variable', 'other_dx_code'])
y = dat['F53_label']

clf.fit(X, y)
print_best_score(clf)

### 2. prepare the X and y for different labeling strategies

In [ ]:
Xs, ys = [], []

cols_to_drop = ['F53_label', 'edinburgh_max','phq9_total_max', 'birthid', 'target_variable', 'other_dx_code']

# 1. F53 only
Xs.append(dat.drop(columns=cols_to_drop))
ys.append(dat['F53_label'])

# 2. edinburgh_max only
dat_selected = dat[dat['edinburgh_max'].isna() == False].copy()
Xs.append(dat_selected.drop(columns=cols_to_drop))
ys.append(np.where(dat_selected['edinburgh_max'] >= 10, 1, 0))

# 3. phq9_total_max only (a lot of nan in this case)
dat_selected = dat[dat['phq9_total_max'].isna() == False].copy()
Xs.append(dat_selected.drop(columns=cols_to_drop))
ys.append(np.where(dat_selected['phq9_total_max'] >= 10, 1, 0))

# 4. edinburgh or phq9
dat_selected = dat[ (dat['edinburgh_max'].isna() == False) | (dat['phq9_total_max'].isna() == False)].copy()
Xs.append(dat_selected.drop(columns=cols_to_drop))
ys.append(np.where( (dat_selected['edinburgh_max'] >= 10) | (dat_selected['phq9_total_max'] >= 10), 1, 0))

# 5. F53 or edinburgh_max or phq9
dat_selected = dat[ (dat['edinburgh_max'].isna() == False) | (dat['phq9_total_max'].isna() == False)].copy()  # this also implies the patients have been scanned
Xs.append(dat_selected.drop(columns=cols_to_drop))
ys.append(np.where( (dat_selected['edinburgh_max'] >= 10) | (dat_selected['phq9_total_max'] >= 10) | (dat_selected['F53_label'] == 1), 1, 0))

# 6. F53 and (edinburgh_max or phq9)
dat_selected = dat[ (dat['edinburgh_max'].isna() == False) | (dat['phq9_total_max'].isna() == False)].copy()  # this also implies the patients have been scanned
Xs.append(dat_selected.drop(columns=cols_to_drop))
ys.append(np.where( ((dat_selected['edinburgh_max'] >= 10) | (dat_selected['phq9_total_max'] >= 10)) & (dat_selected['F53_label'] == 1), 1, 0))

# 7. only apply to patients without any F32, F33, F34, F53 and edinburgh_max < 10, phq9_total_max < 10
cols_todo = [x for x in dat.columns if x.startswith('F32_') or x.startswith('F33_') or x.startswith('F34_1') or x.startswith('F53')]
cols_todo = [x for x in cols_todo if x.endswith('_has_dx')]
condition_1 = dat['phq_or_edinburgh_max_99046-5'] < 10
condition_2 = dat['phq_or_edinburgh_max_21012959'] < 10
condition_3 = dat[cols_todo].sum(axis=1) == 0
dat_selected = dat[condition_1 & condition_2 & condition_3].copy()
dat_selected = dat_selected[ (dat_selected['edinburgh_max'].isna() == False) | (dat_selected['phq9_total_max'].isna() == False)].copy()  # below is the same as 5
Xs.append(dat_selected.drop(columns=cols_to_drop))
ys.append(np.where( (dat_selected['edinburgh_max'] >= 10) | (dat_selected['phq9_total_max'] >= 10) | (dat_selected['F53_label'] == 1), 1, 0))

# 8. use both F53 and other depression ICD10 codes
Xs.append(dat.drop(columns=cols_to_drop))
ys.append(np.where( (dat['F53_label'] > 0) | (dat['other_dx_code'] > 0), 1, 0))

In [ ]:
for X, y in zip(Xs, ys):
    print(len(y), y.mean())

In [ ]:
from sklearn.metrics import recall_score

def get_many_aucs(X, y, nseeds=1):
    aucs, specs, sens = [], [], []
    for seed in range(nseeds):
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=seed)
        
        clf.verbose = 0
        clf.fit(X_train, y_train)

        y_pred_prob = clf.predict_proba(X_test)
        y_pred = clf.predict(X_test)
        aucs.append(roc_auc_score(y_test, y_pred_prob[:, 1]))
        specs.append(recall_score(y_test, y_pred, pos_label=0))
        sens.append(recall_score(y_test, y_pred))
    
    print('auc= {0:.3f}+-{1:.3f}'.format(np.mean(aucs), np.std(aucs)))
    print('specificity= {0:.3f}+-{1:.3f}'.format(np.mean(specs), np.std(specs)))
    print('sensitivity= {0:.3f}+-{1:.3f}'.format(np.mean(sens), np.std(sens)))
    return aucs

In [ ]:
for X, y in zip(Xs, ys):
    print("=============")
    get_many_aucs(X, y, nseeds=10)

### 3. stick with option 5 and get the feature importances

In [ ]:
dat_selected = dat[ (dat['edinburgh_max'].isna() == False) | (dat['phq9_total_max'].isna() == False)].copy()  # this also implies the patients have been scanned
X = dat_selected.drop(columns=cols_to_drop)
y = (np.where( (dat_selected['edinburgh_max'] >= 10) | (dat_selected['phq9_total_max'] >= 10) | (dat_selected['F53_label'] == 1), 1, 0))
clf.fit(X, y)

In [ ]:
clf.best_params_, clf.best_score_

In [ ]:
from ml_utils import get_shap

shap_value = get_shap(clf.best_estimator_, X)